In [3]:
import os
import math
import sqlite3
from   ucimlrepo import fetch_ucirepo
from   sklearn.model_selection import train_test_split
from   itertools import chain, combinations

def powerset(iterable):
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

path = 'wine.db'
conn = sqlite3.connect('wine.db')


In [4]:
check_table = '''SELECT EXISTS (SELECT name FROM sqlite_schema WHERE  type='table' AND  name='X_train');'''

conn = sqlite3.connect('wine.db')
cursor = conn.cursor()
if cursor.execute(check_table).fetchone()[0] == 0:
    wine_quality = fetch_ucirepo(id=186)['data']['original']
    wine_quality.insert(0, "id", range(1, len(wine_quality) + 1))
    X_train, X_test = train_test_split(wine_quality, test_size=0.2, random_state=42)
    X_train.to_sql('X_train', conn, if_exists='replace', index=True)
    X_test.to_sql('X_test', conn, if_exists='replace', index=True)

In [5]:
player_names = ['fixed_acidity', 'volatile_acidity', 'citric_acid','residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']

path, dist = dict(), dict()

start_id, end_id = 441, 934

for player_name in player_names:

    get_var   = f'''select avg(power(({player_name} - (select avg({player_name}) from X_train)),2)) from X_train'''
    get_value = f'''select {player_name} from X_test where id = :unique_id'''

    var       = cursor.execute(get_var).fetchone()[0]
    start_val = cursor.execute(get_value, {'unique_id': start_id}).fetchone()[0]
    end_val   =  cursor.execute(get_value, {'unique_id': end_id}).fetchone()[0]

    path[player_name] = [start_val, end_val]
    dist[player_name+'_stddev'] = math.sqrt(var)

    print(player_name.rjust(20),
           round( math.sqrt(var),3),
          str(start_val).ljust(6),
          str(end_val).ljust(6),
          str(round((end_val- start_val)/math.sqrt(var),6)).ljust(11))


       fixed_acidity 1.288 8.9    8.9    0.0        
    volatile_acidity 0.162 0.62   0.62   0.0        
         citric_acid 0.145 0.18   0.19   0.069123   
      residual_sugar 4.796 3.8    3.9    0.02085    
           chlorides 0.035 0.176  0.17   -0.172284  
 free_sulfur_dioxide 17.438 52.0   51.0   -0.057346  
total_sulfur_dioxide 56.137 145.0  148.0  0.053441   
             density 0.003 0.9986 0.9986 0.0        
                  pH 0.16 3.16   3.17   0.062652   
           sulphates 0.149 0.88   0.93   0.336369   
             alcohol 1.191 9.2    9.2    0.0        
             quality 0.877 5      5      0.0        


In [6]:
def value(coalition):

    if len(coalition)==0:
        target = {player_name: path[player_name][0]  for player_name in player_names }
    else:
        target = {player_name: path[player_name][player_name in coalition]  for player_name in player_names }

    params = target | dist

    query = f'''select quality
                from X_train
                order by    power((fixed_acidity - :fixed_acidity)/:fixed_acidity_stddev ,2)
                          + power((volatile_acidity - :volatile_acidity)/:volatile_acidity_stddev ,2)
                          + power((citric_acid - :citric_acid)/:citric_acid_stddev ,2)
                          + power((residual_sugar - :residual_sugar)/:residual_sugar_stddev ,2)
                          + power((chlorides - :chlorides)/:chlorides_stddev ,2)
                          + power((free_sulfur_dioxide - :free_sulfur_dioxide)/:free_sulfur_dioxide_stddev ,2)
                          + power((total_sulfur_dioxide - :total_sulfur_dioxide)/:total_sulfur_dioxide ,2)
                          + power((density - :density)/ :density_stddev ,2)
                          + power((pH - :pH)/:pH_stddev,2)
                          + power((sulphates - :sulphates)/:sulphates_stddev,2)
                          + power((alcohol - :alcohol)/:alcohol_stddev,2) '''

    return cursor.execute(query, params).fetchone()[0]




In [7]:
def gamma(players, coalition):
    N = len(players)
    S = len(coalition)
    return math.factorial(S) * math.factorial(N - S - 1) / math.factorial(N)

player_set = set(player_names) - {'quality'}
phi = 0
for player in player_set:
    coalitions = player_set - {player}
    phi_i=0.0
    for S in powerset(coalitions):
        v2 = value(set(S).union({player}))
        v1 = value(set(S))
        phi_i += gamma(player_set, S) * (v2 - v1)

    print(player, round(phi_i,3))

    phi += phi_i

print(phi)


pH 0.0
sulphates 0.0
chlorides 0.0
fixed_acidity 0.0
volatile_acidity 0.0
density 0.0
citric_acid 0.0
total_sulfur_dioxide 0.0
alcohol 0.0
free_sulfur_dioxide 0.0
residual_sugar 0.0
0.0


In [8]:
conn.close()

In [6]:
params

NameError: name 'params' is not defined